In [ ]:
import importlib, subprocess, sys
REQUIRED = {'ultralytics': 'ultralytics>=8.3.0', 'cv2': 'opencv-python', 'tqdm': 'tqdm', 'pandas': 'pandas', 'requests': 'requests'}
missing = []
for mod, pkg in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    print('Installing missing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

In [ ]:
import os, json, glob, shutil, time, random, math, csv, urllib.request, zipfile, tarfile, hashlib, subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

random.seed(0); np.random.seed(0); torch.manual_seed(0)

device = 0 if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
print('Device:', device)

In [ ]:
LAB = Path(os.getcwd()).resolve()
if LAB.name != 'lab-6':
    cand = LAB / 'lab-6'
    if cand.exists():
        LAB = cand.resolve()
DATA = LAB / 'data'
RTSD_DIR = DATA / 'rtsd'
YOLO_DIR = DATA / 'rtsd_yolo'
REAL_DIR = DATA / 'real_photos'
RUNS_DIR = LAB / 'runs'
RESULTS_PATH = LAB / 'results.json'

for p in [RTSD_DIR, YOLO_DIR/'images'/'train', YOLO_DIR/'images'/'val', YOLO_DIR/'labels'/'train', YOLO_DIR/'labels'/'val']:
    p.mkdir(parents=True, exist_ok=True)
REAL_DIR.mkdir(parents=True, exist_ok=True)

# 8 super-classes by ПДД category (first digit of sign code)
SUPER_NAMES = ['warning', 'priority', 'prohibitory', 'mandatory', 'special', 'informational', 'service', 'additional']
SUPER_RU = ['Предупреждающие', 'Приоритета', 'Запрещающие', 'Предписывающие', 'Особых_предписаний', 'Информационные', 'Сервиса', 'Доп_информации']
NUM_CLASSES = len(SUPER_NAMES)

def sign_to_super(sign_class):
    """Map RTSD sign code (e.g. '1.1', '2.4', '3.24') to super-class id 0..7 by first digit."""
    s = str(sign_class).strip()
    if not s or s.lower() == 'nan':
        return -1
    head = s.split('.')[0].split('_')[0]
    try:
        d = int(head)
        return d - 1 if 1 <= d <= 8 else -1
    except ValueError:
        return -1

print('LAB:', LAB)
for i, (n, ru) in enumerate(zip(SUPER_NAMES, SUPER_RU)):
    print(f'  {i}  {n:14}  {ru}')

In [ ]:
# Скачивание RTSD. Поддерживается:
#   1) kaggle CLI (если ~/.kaggle/kaggle.json настроен)
#   2) ENV var RTSD_URL (прямая ссылка на zip)
#   3) Уже распакованный руками датасет в data/rtsd/

def find_rtsd():
    """Returns (frames_dir, csv_files) or (None, []) if not found."""
    csvs = sorted(Path(RTSD_DIR).rglob('*.csv'))
    best_dir, best_n = None, 0
    for d in Path(RTSD_DIR).rglob('*'):
        if not d.is_dir():
            continue
        n = sum(1 for _ in d.glob('*.jpg')) + sum(1 for _ in d.glob('*.png'))
        if n > best_n:
            best_n = n
            best_dir = d
    if best_n < 100:
        best_dir = None
    return best_dir, csvs

frames_dir, csv_files = find_rtsd()
if frames_dir and csv_files:
    n_frames = sum(1 for _ in frames_dir.glob('*.jpg')) + sum(1 for _ in frames_dir.glob('*.png'))
    print(f'RTSD already present:')
    print(f'  frames: {frames_dir} ({n_frames} images)')
    print(f'  csvs:   {csv_files}')
else:
    print('RTSD not found in', RTSD_DIR)
    kaggle_datasets = [
        'watchman/rtsd-dataset',
        'watchman/rtsd-public',
    ]
    downloaded = False
    for slug in kaggle_datasets:
        print(f'\nTrying kaggle: {slug}')
        try:
            r = subprocess.run(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(RTSD_DIR), '--unzip'], capture_output=True, text=True, timeout=3600)
            if r.returncode == 0:
                print('Kaggle download OK')
                downloaded = True
                break
            else:
                print(f'  failed: {r.stderr[:200]}')
        except (subprocess.CalledProcessError, FileNotFoundError, subprocess.TimeoutExpired) as e:
            print(f'  error: {e}')
    if not downloaded:
        url = os.environ.get('RTSD_URL', '')
        if url:
            print(f'\nDownloading from RTSD_URL = {url}')
            arch = RTSD_DIR / 'rtsd_archive'
            urllib.request.urlretrieve(url, arch)
            try:
                with zipfile.ZipFile(arch) as z: z.extractall(RTSD_DIR)
            except zipfile.BadZipFile:
                with tarfile.open(arch) as t: t.extractall(RTSD_DIR)
            arch.unlink()
            downloaded = True
    frames_dir, csv_files = find_rtsd()
    if not (frames_dir and csv_files):
        print('\n=== ВНИМАНИЕ: RTSD не скачан автоматически ===')
        print('Скачайте вручную одним из способов:')
        print('  A) Kaggle:')
        print('     pip install kaggle && поместите kaggle.json в ~/.kaggle/ (chmod 600)')
        print(f'     kaggle datasets download -d watchman/rtsd-dataset -p {RTSD_DIR} --unzip')
        print('  B) МГУ (https://graphics.cs.msu.ru/projects/traffic-sign-recognition.html)')
        print(f'     wget <ссылка-на-rtsd-r1.tar> -O {RTSD_DIR}/rtsd.tar')
        print(f'     tar -xf {RTSD_DIR}/rtsd.tar -C {RTSD_DIR}/')
        print('  C) Прямая ссылка через ENV:')
        print('     export RTSD_URL="https://...rtsd.zip" && перезапустить эту ячейку')
        print(f'\nОжидаемая структура внутри {RTSD_DIR}:')
        print('  • директория с *.jpg или *.png (минимум 100 файлов)')
        print('  • один или несколько *.csv с колонками filename + bbox + sign_class')
        raise SystemExit('RTSD download required before running further cells')
    else:
        print(f'\nNow have:')
        print(f'  frames: {frames_dir}')
        print(f'  csvs:   {csv_files}')

In [ ]:
# Парсинг CSV-аннотаций RTSD. Гибкое сопоставление колонок (разные версии RTSD имеют разный schema).
def parse_rtsd_csv(csv_path):
    df = pd.read_csv(csv_path)
    cols = {c.lower().strip(): c for c in df.columns}
    fname = next((cols[k] for k in ['filename', 'file_name', 'fname', 'image', 'image_name', 'image_path', 'frame'] if k in cols), None)
    cls = next((cols[k] for k in ['sign_class', 'class', 'category', 'label', 'sign'] if k in cols), None)
    fmt = None; xc = yc = wc = hc = None
    if 'x_from' in cols and 'width' in cols:
        xc, yc, wc, hc, fmt = cols['x_from'], cols['y_from'], cols['width'], cols['height'], 'xywh'
    elif 'xtl' in cols and 'xbr' in cols:
        xc, yc, wc, hc, fmt = cols['xtl'], cols['ytl'], cols['xbr'], cols['ybr'], 'xyxy'
    elif 'x_min' in cols and 'x_max' in cols:
        xc, yc, wc, hc, fmt = cols['x_min'], cols['y_min'], cols['x_max'], cols['y_max'], 'xyxy'
    elif 'x' in cols and 'w' in cols:
        xc, yc, wc, hc, fmt = cols['x'], cols['y'], cols['w'], cols['h'], 'xywh'
    else:
        raise RuntimeError(f'cannot infer bbox columns from {list(df.columns)}')
    if not fname or not cls:
        raise RuntimeError(f'cannot find filename/class columns in {list(df.columns)}')
    sub = df[[fname, xc, yc, wc, hc, cls]].copy()
    sub.columns = ['filename', 'a', 'b', 'c', 'd', 'sign_class']
    if fmt == 'xywh':
        sub['x1'] = sub['a'].astype(float); sub['y1'] = sub['b'].astype(float)
        sub['x2'] = sub['x1'] + sub['c'].astype(float); sub['y2'] = sub['y1'] + sub['d'].astype(float)
    else:
        sub['x1'] = sub[['a', 'c']].min(axis=1).astype(float); sub['x2'] = sub[['a', 'c']].max(axis=1).astype(float)
        sub['y1'] = sub[['b', 'd']].min(axis=1).astype(float); sub['y2'] = sub[['b', 'd']].max(axis=1).astype(float)
    sub = sub[['filename', 'x1', 'y1', 'x2', 'y2', 'sign_class']]
    sub['super'] = sub['sign_class'].apply(sign_to_super)
    sub = sub[sub['super'] >= 0].reset_index(drop=True)
    return sub

all_dfs = []
for cp in csv_files:
    try:
        d = parse_rtsd_csv(cp)
        print(f'parsed {cp.name}: {len(d)} valid rows')
        all_dfs.append(d)
    except Exception as e:
        print(f'skip {cp.name}: {e}')
if not all_dfs:
    raise RuntimeError('no usable CSVs parsed')
ann = pd.concat(all_dfs, ignore_index=True).drop_duplicates()
print(f'\nTotal annotations: {len(ann)}')
print('Per super-class:')
for c, n in ann['super'].value_counts().sort_index().items():
    print(f'  {c} {SUPER_NAMES[c]:14} {n}')

In [ ]:
# Сопоставление имён файлов с реально существующими картинками (RTSD кладёт frames в подпапку).
frame_index = {}
for ext in ('*.jpg', '*.JPG', '*.jpeg', '*.png', '*.PNG'):
    for p in Path(frames_dir).rglob(ext):
        frame_index[p.name] = p
        frame_index[p.stem] = p
print(f'Frames index: {len(frame_index)} unique entries')

def resolve_image_path(filename):
    s = str(filename)
    bn = os.path.basename(s)
    if bn in frame_index:
        return frame_index[bn]
    stem = os.path.splitext(bn)[0]
    if stem in frame_index:
        return frame_index[stem]
    return None

ann['path'] = ann['filename'].apply(lambda x: resolve_image_path(x))
missing = ann['path'].isna().sum()
print(f'Annotations with image not found on disk: {missing} (will be dropped)')
ann = ann.dropna(subset=['path']).reset_index(drop=True)
print(f'Annotations after image existence filter: {len(ann)}')

In [ ]:
# Сабсет: 30 000 train + 1 500 val. Стратифицировано: для val — picks по ~VAL/8 от каждого супер-класса.
TRAIN_TARGET = 30000
VAL_TARGET = 1500

imgs = ann.groupby('filename').agg(supers=('super', lambda s: sorted(set(s))), n=('super', 'size')).reset_index()
print(f'Unique images with signs: {len(imgs)}')

if len(imgs) < TRAIN_TARGET + VAL_TARGET:
    print(f'WARN: only {len(imgs)} unique images available; TRAIN_TARGET+VAL_TARGET={TRAIN_TARGET+VAL_TARGET}')
    VAL_TARGET = min(VAL_TARGET, len(imgs) // 20)
    TRAIN_TARGET = max(0, len(imgs) - VAL_TARGET)
    print(f'Adjusted: TRAIN={TRAIN_TARGET}, VAL={VAL_TARGET}')

imgs_shuf = imgs.sample(frac=1, random_state=0).reset_index(drop=True)

val_per_class = max(1, VAL_TARGET // NUM_CLASSES)
val_set = set()
for sc in range(NUM_CLASSES):
    cand = imgs_shuf[imgs_shuf['supers'].apply(lambda lst: sc in lst)]
    take = cand['filename'].head(val_per_class).tolist()
    val_set.update(take)
while len(val_set) < VAL_TARGET:
    extra = imgs_shuf[~imgs_shuf['filename'].isin(val_set)]['filename'].head(VAL_TARGET - len(val_set)).tolist()
    val_set.update(extra)
    if not extra: break
val_set = set(list(val_set)[:VAL_TARGET])

rest = [f for f in imgs_shuf['filename'].tolist() if f not in val_set]
train_set = set(rest[:TRAIN_TARGET])

print(f'TRAIN: {len(train_set)} images')
print(f'VAL:   {len(val_set)} images')

def class_dist(s):
    cc = Counter()
    for sup_list in imgs_shuf[imgs_shuf['filename'].isin(s)]['supers']:
        for c in sup_list: cc[c] += 1
    return cc
print('\nClass dist (per-image, многосчётно если на одной картинке несколько классов):')
tr_dist = class_dist(train_set); va_dist = class_dist(val_set)
for c in range(NUM_CLASSES):
    print(f'  {c} {SUPER_NAMES[c]:14}  train={tr_dist[c]:>6}  val={va_dist[c]:>4}')

In [ ]:
# Конвертация в YOLO seg. bbox -> прямоугольный полигон (4 угла), нормированные в [0,1].
def link_or_copy(src, dst):
    src, dst = str(src), str(dst)
    if os.path.exists(dst): return
    try: os.symlink(src, dst); return
    except (OSError, NotImplementedError): pass
    try: os.link(src, dst); return
    except OSError: pass
    shutil.copy2(src, dst)

def write_yolo_seg(img_path, sub_df, out_lbl_path):
    img = cv2.imread(str(img_path))
    if img is None: return False, 0
    H, W = img.shape[:2]
    lines = []
    for _, r in sub_df.iterrows():
        cls = int(r['super'])
        x1, y1, x2, y2 = float(r['x1']), float(r['y1']), float(r['x2']), float(r['y2'])
        x1 = max(0, min(W-1, x1)); x2 = max(0, min(W-1, x2))
        y1 = max(0, min(H-1, y1)); y2 = max(0, min(H-1, y2))
        if x2 - x1 < 2 or y2 - y1 < 2: continue
        pts = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
        coords = []
        for x, y in pts:
            coords.append(min(max(x / W, 0.0), 1.0))
            coords.append(min(max(y / H, 0.0), 1.0))
        lines.append(f'{cls} ' + ' '.join(f'{c:.6f}' for c in coords))
    out_lbl_path = Path(out_lbl_path)
    out_lbl_path.parent.mkdir(parents=True, exist_ok=True)
    out_lbl_path.write_text('\n'.join(lines))
    return True, len(lines)

ann_by_file = ann.groupby('filename')

def convert_subset(filenames, img_dst, lbl_dst):
    n_ok, n_skip, n_inst = 0, 0, 0
    for fn in tqdm(filenames):
        sub = ann_by_file.get_group(fn) if fn in ann_by_file.groups else None
        if sub is None or len(sub) == 0:
            n_skip += 1; continue
        src = sub.iloc[0]['path']
        out_lbl = Path(lbl_dst) / (Path(fn).stem + '.txt')
        ok, cnt = write_yolo_seg(src, sub, out_lbl)
        if not ok: n_skip += 1; continue
        link_or_copy(src, Path(img_dst) / Path(fn).name)
        n_ok += 1; n_inst += cnt
    return n_ok, n_skip, n_inst

t0 = time.time()
tr_ok, tr_skip, tr_inst = convert_subset(list(train_set), YOLO_DIR/'images'/'train', YOLO_DIR/'labels'/'train')
va_ok, va_skip, va_inst = convert_subset(list(val_set), YOLO_DIR/'images'/'val', YOLO_DIR/'labels'/'val')
print(f'\nTRAIN: ok={tr_ok}, skipped={tr_skip}, instances={tr_inst}')
print(f'VAL:   ok={va_ok}, skipped={va_skip}, instances={va_inst}')
print(f'Conversion {time.time()-t0:.1f}s')

In [ ]:
data_yaml = YOLO_DIR / 'data.yaml'
with open(data_yaml, 'w') as f:
    f.write(f'path: {YOLO_DIR}\n')
    f.write('train: images/train\n')
    f.write('val: images/val\n')
    f.write(f'nc: {NUM_CLASSES}\n')
    f.write('names: ' + json.dumps(SUPER_NAMES) + '\n')
print(open(data_yaml).read())

In [ ]:
# Обучение YOLO11s-seg.
# - epochs=80, patience=20
# - imgsz=640, batch=16 (RTX 2080S, ~6 GB VRAM)
# - cache='disk' — 30K картинок в RAM могут не влезть
# - save_period=3 — checkpoint каждые 3 эпохи (≈ 5K итераций при 30K/16=1875 iter/epoch)
#   плюс last.pt после каждой эпохи + best.pt при улучшении
# - amp=True — fp16 (Turing tensor cores)
from ultralytics import YOLO

MODEL_NAME = 'yolo11s-seg.pt'
EPOCHS = 80
IMG_SIZE = 640
BATCH = 16
PATIENCE = 20
SAVE_PERIOD = 3
RUN_NAME = 'rtsd_yolo11s'

model = YOLO(MODEL_NAME)
print('Loaded:', MODEL_NAME)

t_start = time.time()
train_results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    cache='disk',
    amp=True,
    device=device,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=True,
    seed=0,
    save_period=SAVE_PERIOD,
    workers=4,
)
training_seconds = time.time() - t_start
print(f'\nTraining wall time: {training_seconds/60:.1f} minutes')
best_pt = Path(train_results.save_dir) / 'weights' / 'best.pt'
print('Best weights:', best_pt)

In [ ]:
best_model = YOLO(str(best_pt))
val_metrics = best_model.val(data=str(data_yaml), split='val', imgsz=IMG_SIZE, batch=BATCH, device=device, plots=False, verbose=False)
print('Box mAP50    :', round(float(val_metrics.box.map50), 4))
print('Box mAP50-95 :', round(float(val_metrics.box.map), 4))
print('Mask mAP50   :', round(float(val_metrics.seg.map50), 4))
print('Mask mAP50-95:', round(float(val_metrics.seg.map), 4))
print('\nPer-class mask AP50:')
if hasattr(val_metrics.seg, 'ap50') and len(val_metrics.seg.ap50) > 0:
    for i, ap in enumerate(val_metrics.seg.ap50):
        print(f'  {i} {SUPER_NAMES[i]:14}  AP50={float(ap):.3f}')

In [ ]:
# Кастомные метрики (как в ТЗ): IoU, Precision, Recall, L2, %IoU≥{0.5,0.75,0.9}
def load_yolo_seg_gt(label_path, H, W):
    masks, classes = [], []
    if not os.path.exists(label_path): return masks, classes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7: continue
            cls = int(parts[0])
            coords = np.asarray([float(x) for x in parts[1:]]).reshape(-1, 2)
            coords[:, 0] *= W; coords[:, 1] *= H
            poly = coords.astype(np.int32)
            m = np.zeros((H, W), dtype=np.uint8)
            cv2.fillPoly(m, [poly], 1)
            masks.append(m.astype(bool)); classes.append(cls)
    return masks, classes

def predict_masks(model, img_path, imgsz=640, conf=0.25):
    res = model.predict(str(img_path), imgsz=imgsz, conf=conf, verbose=False, device=device)[0]
    H, W = res.orig_shape
    masks, classes, scores = [], [], []
    if res.masks is not None and len(res.masks) > 0:
        m_arr = res.masks.data.cpu().numpy()
        cls_arr = res.boxes.cls.cpu().numpy().astype(int)
        scr_arr = res.boxes.conf.cpu().numpy()
        for k in range(m_arr.shape[0]):
            m = m_arr[k]
            if m.shape != (H, W):
                m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
            masks.append(m.astype(bool)); classes.append(int(cls_arr[k])); scores.append(float(scr_arr[k]))
    return masks, classes, scores, (H, W)

def iou_pair(a, b):
    inter = np.logical_and(a, b).sum(); union = np.logical_or(a, b).sum()
    return float(inter)/float(union) if union > 0 else 0.0

def l2_pair(a, b):
    diff = a.astype(np.float32) - b.astype(np.float32)
    return float(np.sqrt((diff*diff).sum()))

def evaluate(model, img_dir, lbl_dir, num_classes, match_iou=0.5, conf=0.25, imgsz=640, max_imgs=None, require_gt=False):
    img_files = sorted(
        glob.glob(str(Path(img_dir) / '*.jpg')) + glob.glob(str(Path(img_dir) / '*.jpeg')) +
        glob.glob(str(Path(img_dir) / '*.png')) + glob.glob(str(Path(img_dir) / '*.JPG')) +
        glob.glob(str(Path(img_dir) / '*.PNG'))
    )
    if max_imgs: img_files = img_files[:max_imgs]
    if require_gt:
        img_files = [ip for ip in img_files
                     if (Path(lbl_dir)/(Path(ip).stem+'.txt')).exists()
                     and (Path(lbl_dir)/(Path(ip).stem+'.txt')).stat().st_size > 0]
    tp = fp = fn = 0
    iou_per_match, l2_per_match, image_mean_iou = [], [], []
    for ip in tqdm(img_files, desc=f'Eval {Path(img_dir).name}'):
        H, W = cv2.imread(ip).shape[:2]
        gt_m, gt_c = load_yolo_seg_gt(Path(lbl_dir)/(Path(ip).stem+'.txt'), H, W)
        pr_m, pr_c, pr_s, _ = predict_masks(model, ip, imgsz=imgsz, conf=conf)
        used_pr, used_gt = set(), set()
        per_image_ious = []
        for c in range(num_classes):
            gi = [i for i, x in enumerate(gt_c) if x == c]
            pi = [i for i, x in enumerate(pr_c) if x == c]
            if not gi or not pi: continue
            mat = np.zeros((len(pi), len(gi)))
            for u, p in enumerate(pi):
                for v, g in enumerate(gi):
                    mat[u, v] = iou_pair(pr_m[p], gt_m[g])
            while True:
                u, v = np.unravel_index(mat.argmax(), mat.shape)
                if mat[u, v] <= 0: break
                p, g = pi[u], gi[v]
                iv = float(mat[u, v])
                if iv >= match_iou:
                    tp += 1; used_pr.add(p); used_gt.add(g)
                    iou_per_match.append(iv); l2_per_match.append(l2_pair(pr_m[p], gt_m[g]))
                    per_image_ious.append(iv)
                mat[u, :] = -1; mat[:, v] = -1
        fp += len([i for i in range(len(pr_c)) if i not in used_pr])
        fn += len([i for i in range(len(gt_c)) if i not in used_gt])
        if len(gt_m) > 0:
            image_mean_iou.append(float(np.mean(per_image_ious)) if per_image_ious else 0.0)
    precision = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    recall = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    return {
        'images': len(img_files),
        'tp': tp, 'fp': fp, 'fn': fn,
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'mean_iou_matched': round(float(np.mean(iou_per_match)), 4) if iou_per_match else 0.0,
        'mean_l2_matched': round(float(np.mean(l2_per_match)), 2) if l2_per_match else 0.0,
        'frac_iou_ge_0.50': round(float(np.mean([x>=0.50 for x in image_mean_iou])), 4) if image_mean_iou else 0.0,
        'frac_iou_ge_0.75': round(float(np.mean([x>=0.75 for x in image_mean_iou])), 4) if image_mean_iou else 0.0,
        'frac_iou_ge_0.90': round(float(np.mean([x>=0.90 for x in image_mean_iou])), 4) if image_mean_iou else 0.0,
    }

val_custom = evaluate(best_model, YOLO_DIR/'images'/'val', YOLO_DIR/'labels'/'val', NUM_CLASSES)
for k, v in val_custom.items(): print(f'{k:>20}: {v}')

In [ ]:
# Качественная визуализация: GT vs prediction на 4 случайных val
def overlay(img_rgb, masks, classes, scores=None, alpha=0.45):
    rng_p = np.random.default_rng(123)
    palette = rng_p.integers(60, 255, size=(NUM_CLASSES, 3), dtype=np.int32)
    out = img_rgb.copy()
    for k, (m, c) in enumerate(zip(masks, classes)):
        color = palette[c % NUM_CLASSES]
        layer = out.copy(); layer[m] = color
        out = cv2.addWeighted(out, 1-alpha, layer, alpha, 0)
        ys, xs = np.where(m)
        if len(xs):
            x0, y0 = int(xs.min()), int(ys.min())
            label = SUPER_NAMES[c] if 0 <= c < NUM_CLASSES else f'cls{c}'
            if scores is not None and k < len(scores): label += f' {scores[k]:.2f}'
            cv2.putText(out, label, (x0, max(15, y0-4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, tuple(int(v) for v in color), 1)
    return out

val_imgs_paths = sorted(glob.glob(str(YOLO_DIR/'images'/'val'/'*.jpg')) + glob.glob(str(YOLO_DIR/'images'/'val'/'*.png')))
if val_imgs_paths:
    rng2 = random.Random(42)
    sample = rng2.sample(val_imgs_paths, min(4, len(val_imgs_paths)))
    fig, axes = plt.subplots(len(sample), 2, figsize=(12, 4*len(sample)))
    if len(sample) == 1: axes = axes.reshape(1, 2)
    for r, ip in enumerate(sample):
        img_rgb = cv2.imread(ip)[:, :, ::-1]
        H, W = img_rgb.shape[:2]
        gt_m, gt_c = load_yolo_seg_gt(YOLO_DIR/'labels'/'val'/f'{Path(ip).stem}.txt', H, W)
        pr_m, pr_c, pr_s, _ = predict_masks(best_model, ip)
        axes[r, 0].imshow(overlay(img_rgb, gt_m, gt_c)); axes[r, 0].set_title(f'GT  {Path(ip).name}'); axes[r, 0].axis('off')
        axes[r, 1].imshow(overlay(img_rgb, pr_m, pr_c, pr_s)); axes[r, 1].set_title('Pred'); axes[r, 1].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Тест на самостоятельных уличных фото из data/real_photos/
real_imgs = sorted(
    glob.glob(str(REAL_DIR/'*.jpg')) + glob.glob(str(REAL_DIR/'*.jpeg')) +
    glob.glob(str(REAL_DIR/'*.png')) + glob.glob(str(REAL_DIR/'*.JPG')) + glob.glob(str(REAL_DIR/'*.PNG'))
)
real_metrics = None
if not real_imgs:
    print(f'[skip] нет фото в {REAL_DIR}')
else:
    print(f'Found {len(real_imgs)} real photo(s)')
    real_lbl_dir = REAL_DIR/'labels'; real_lbl_dir.mkdir(exist_ok=True)
    real_have_gt = 0
    for ip in real_imgs:
        existing_txt = Path(ip).with_suffix('.txt')
        out_txt = real_lbl_dir / (Path(ip).stem + '.txt')
        if existing_txt.exists():
            shutil.copy2(existing_txt, out_txt)
            if out_txt.stat().st_size > 0: real_have_gt += 1
        elif not out_txt.exists():
            out_txt.write_text('')
    pred_dir = REAL_DIR/'predictions'; pred_dir.mkdir(exist_ok=True)
    n = len(real_imgs)
    fig, axes = plt.subplots(n, 1, figsize=(10, 5*n))
    if n == 1: axes = [axes]
    for i, ip in enumerate(real_imgs):
        img_rgb = cv2.imread(ip)[:, :, ::-1]
        pr_m, pr_c, pr_s, _ = predict_masks(best_model, ip)
        vis = overlay(img_rgb, pr_m, pr_c, pr_s)
        cv2.imwrite(str(pred_dir/Path(ip).name), vis[:, :, ::-1])
        axes[i].imshow(vis); axes[i].set_title(f'{Path(ip).name} — {len(pr_m)} det'); axes[i].axis('off')
    plt.tight_layout(); plt.show()
    if real_have_gt > 0:
        print(f'Computing metrics on {real_have_gt} annotated photo(s)...')
        real_metrics = evaluate(best_model, REAL_DIR, real_lbl_dir, NUM_CLASSES, require_gt=True)
        for k, v in real_metrics.items(): print(f'  {k:>20}: {v}')
    else:
        print('Без разметки метрики не считаем (custom_metrics_real_photos = null).')

In [ ]:
results = {
    'task': 'instance_segmentation_8_road_sign_categories',
    'dataset': 'RTSD (MSU Graphics) — 8 super-classes by ПДД category',
    'super_classes': SUPER_NAMES,
    'super_classes_ru': SUPER_RU,
    'mask_strategy': 'bbox_rectangle',
    'model': 'YOLO11s-seg (pretrained COCO, fine-tuned)',
    'num_classes': NUM_CLASSES,
    'train_size': tr_ok,
    'val_size': va_ok,
    'train_instances': tr_inst,
    'val_instances': va_inst,
    'hyperparameters': {
        'imgsz': IMG_SIZE, 'batch': BATCH, 'epochs': EPOCHS, 'patience': PATIENCE,
        'save_period': SAVE_PERIOD, 'optimizer': 'auto', 'amp': True, 'cache': 'disk',
        'device': str(device),
    },
    'training_seconds': round(training_seconds, 1),
    'training_minutes': round(training_seconds/60, 1),
    'best_weights': str(best_pt),
    'ultralytics_metrics': {
        'box_mAP50': round(float(val_metrics.box.map50), 4),
        'box_mAP50_95': round(float(val_metrics.box.map), 4),
        'mask_mAP50': round(float(val_metrics.seg.map50), 4),
        'mask_mAP50_95': round(float(val_metrics.seg.map), 4),
    },
    'custom_metrics_val': val_custom,
    'custom_metrics_real_photos': real_metrics,
}
with open(RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(json.dumps(results, indent=2, ensure_ascii=False))